# 02 · Modelos clásicos (estado y fruta)

Fase CRISP-DM: **Modelado**.

Los modelos clásicos no ven la foto: ven **359 números** por recorte (`src/data/features.py`):

| Grupo | Cuántos | Qué mide |
|---|---|---|
| Color | 96 | histograma de 16 barras de R, G, B, H, S y V, solo sobre la fruta |
| Textura (LBP) | 256 | cómo cambia el brillo entre cada píxel y sus 8 vecinos: manchas, arrugas, moho |
| Forma | 7 | proporciones, circularidad, alargamiento: ayudan a saber qué fruta es |

Se entrenan 4 modelos (`src/models/model_definitions.py`): regresión logística, SVM lineal y Random Forest para el **estado**, y un Random Forest para la **fruta**.

**Cómo se eligen los hiperparámetros.** `GridSearchCV` prueba todas las combinaciones de la rejilla con **validación cruzada de 3 partes por grupos** (`StratifiedGroupKFold`): en cada vuelta entrena con 2/3 de los grupos de train y mide en el tercio restante, así que nunca mide con fotos "hermanas" de las de entrenamiento. Gana la combinación con mejor F1 macro promedio. Val no participa en la elección.

Para entrenar (tarda unos minutos):
```
python -m src.main entrenar --solo clasicos
```
Este notebook solo **lee** lo que dejó ese comando en `results/entrenamiento_clasicos.json`.

In [ ]:
from pathlib import Path
import json
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "src" / "config.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import pandas as pd

from src import config

def read(name):
    """Lee un resultado de results/; si no existe, explica qué comando lo genera."""
    path = config.RESULTS / name
    if not path.is_file():
        raise FileNotFoundError(f"Falta {path}. Corre primero los pasos del README.")
    return json.loads(path.read_text(encoding="utf-8"))

config.FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
report = read("entrenamiento_clasicos.json")
summary = pd.DataFrame([
    {"modelo": name, "predice": item["target"], "mejores hiperparámetros": item["best_params"],
     "F1 CV (train)": round(item["cv_f1_mean"], 3), "± desv.": round(item["cv_f1_std"], 3),
     "F1 val": round(item["validation"]["f1_macro"], 3), "MB": item["size_mb"]}
    for name, item in report["models"].items()])
summary

## Todas las combinaciones probadas

Cada barra es una combinación de la rejilla; la línea vertical es su desviación entre las 3 partes. Si dos combinaciones están a menos de una desviación, la diferencia no es importante.

In [ ]:
fig, axes = plt.subplots(1, len(report["models"]), figsize=(5 * len(report["models"]), 4))
for ax, (name, item) in zip(axes, report["models"].items()):
    grid = pd.DataFrame(item["grid"]).sort_values("cv_f1_mean")
    labels = [", ".join(f"{k.split('__')[-1]}={v}" for k, v in p.items()) for p in grid["params"]]
    ax.barh(labels, grid["cv_f1_mean"], xerr=grid["cv_f1_std"], color="#4c78a8")
    ax.set_title(name)
    ax.set_xlabel("F1 macro en validación cruzada")
    ax.tick_params(axis="y", labelsize=7)
fig.tight_layout()
fig.savefig(config.FIGURES / "rejillas_clasicos.png", dpi=150)

## ¿Memoriza? Train contra validación cruzada

Si el F1 en las partes de entrenamiento es casi 1 y el de validación cruzada es mucho menor, el modelo está memorizando. El Random Forest suele dar 1.0 en train (sus árboles crecen hasta separar todo); lo que importa es su F1 en validación cruzada y en val.

In [ ]:
pd.DataFrame([{"modelo": n, "F1 train": round(i["train_f1_mean"], 3),
               "F1 CV": round(i["cv_f1_mean"], 3)} for n, i in report["models"].items()])

## Errores en val del Random Forest de estado

In [ ]:
rf = report["models"]["random_forest"]["validation"]
pd.DataFrame(rf["confusion_matrix"], index=[f"real {c}" for c in rf["labels"]],
             columns=[f"predicho {c}" for c in rf["labels"]])